<div style="background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%); padding: 30px; border-radius: 8px; margin-bottom: 25px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);">
    <h1 style="color: #f8fafc; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-weight: 700; margin: 0; font-size: 2.2em; letter-spacing: -0.5px;"> Notas Medias PISA 2012-2022 🌍
</h1>
    <p style="color: #94a3b8; font-family: 'Segoe UI', sans-serif; margin: 8px 0 0 0; font-size: 1.1em;"> Generacion de notas media por asignatura y estrato </p>
    
</div>

# Importacion de librerias

In [2]:
import sys
import os
import gc
from pathlib import Path

#Indicamos el workspace principal (dos niveles atrás)
sys.path.append(os.path.abspath('../../'))

%load_ext autoreload
%autoreload 2


In [3]:
#Manipulación y Análisis de Datos
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import plotly.express as px

#Funciones propias estadisticas
from src.utils_pisa import *


In [4]:
ruta_datos_raw = Path('../../data/raw')
ruta_datos_intermedios = Path('../../data/intermediate')
ruta_medias_asignaturas = ruta_datos_intermedios/'medias_asignaturas_anual'

In [5]:
establecer_semilla()

✅ Semillas del proyecto establecidas con éxito (seed=42).


# Medias asignaturas PISA 2022

In [6]:
#Cargamos todos los datos de estudiantes de 2022
ruta_archivo_estudiantes_2022 = ruta_datos_raw / 'pisa2022_estudiantes.parquet'
archivo_pisa_2022 = pq.ParquetFile(ruta_archivo_estudiantes_2022)
columnas_disponibles_2022 = archivo_pisa_2022.schema.names


In [8]:
#Generamos los nombres de las columnas que nos interesan para cada asignatura y el peso del alumno

#Matematias
columnas_math = [col for col in columnas_disponibles_2022 if col.startswith('PV') and col.endswith('MATH')]

#Literatura
columnas_lectura = [col for col in columnas_disponibles_2022 if col.startswith('PV') and col.endswith('READ')]

#Ciencias
columnas_ciencias = [col for col in columnas_disponibles_2022 if col.startswith('PV') and col.endswith('SCIE')]

#Identifiadores
columnas_id = ['CNT', 'STRATUM']

In [9]:
print(f"Columnas de matemáticas encontradas: {len(columnas_math)}")
print(f"Columnas de lectura encontradas: {len(columnas_lectura)}")
print(f"Columnas de ciencias encontradas: {len(columnas_ciencias)}")

Columnas de matemáticas encontradas: 10
Columnas de lectura encontradas: 10
Columnas de ciencias encontradas: 10


In [10]:
# Cargamos país, estrato, PV valores y peso del alumno.
columnas_interes = columnas_id + columnas_math + columnas_lectura + columnas_ciencias + ['W_FSTUWT']

#Hacemos la lectura de los datos de PV valores que nos interesan
df_pisa = pd.read_parquet(ruta_archivo_estudiantes_2022, columns=columnas_interes)

## Notas por Pais

In [17]:
df_medias_math_pais = puntuaciones_pisa_pais(df_pisa, 'MATH', col_peso='W_FSTUWT')
df_medias_lectura_pais = puntuaciones_pisa_pais(df_pisa, 'READ', col_peso='W_FSTUWT')
df_medias_ciencias_pais = puntuaciones_pisa_pais(df_pisa, 'SCIE', col_peso='W_FSTUWT')

In [18]:
display(df_medias_math_pais.head())

,CNT,media_math_pais
0,AUS,487.084259
1,AUT,487.267487
2,BEL,489.486816
3,CAN,496.947906
4,CHE,507.990570


## Notas por estrato

In [19]:
#Calculamo la media ponderada por cada estrato de cada asignatura 
df_medias_math_estrato = puntuaciones_pisa_estratos(df_pisa, 'MATH', col_peso='W_FSTUWT')
df_medias_lectura_estrato = puntuaciones_pisa_estratos(df_pisa, 'READ', col_peso='W_FSTUWT')
df_medias_ciencias_estrato = puntuaciones_pisa_estratos(df_pisa, 'SCIE', col_peso='W_FSTUWT')

In [20]:
display(df_medias_math_estrato.head())

,CNT,STRATUM,media_math_estrato
0,AUS,AUS01,508.695984
1,AUS,AUS02,479.183319
2,AUS,AUS03,540.415649
3,AUS,AUS04,495.911804
4,AUS,AUS05,478.725861


In [94]:
#Guardamos el dataframe en formato parquet
df_medias_math_pais.to_parquet(ruta_medias_asignaturas/'pisa22_medias_math_pais.parquet', index=False)
df_medias_lectura_pais.to_parquet(ruta_medias_asignaturas/'pisa22_medias_read_pais.parquet', index=False)
df_medias_ciencias_pais.to_parquet(ruta_medias_asignaturas/'pisa22_medias_scie_pais.parquet', index=False)
df_medias_math_estrato.to_parquet(ruta_medias_asignaturas/'pisa22_medias_math_estrato.parquet', index=False)
df_medias_lectura_estrato.to_parquet(ruta_medias_asignaturas/'pisa22_medias_read_estrato.parquet', index=False)
df_medias_ciencias_estrato.to_parquet(ruta_medias_asignaturas/'pisa22_medias_scie_estrato.parquet', index=False)

# Medias asignaturas PISA 2018 por país y estrato

In [95]:
#Cargamos todos los datos de estudiantes de 2018
ruta_archivo_estudiantes_2018 = ruta_datos_raw / 'pisa2018_estudiantes.parquet'
archivo_pisa_2018 = pq.ParquetFile(ruta_archivo_estudiantes_2018)
columnas_disponibles_2018 = archivo_pisa_2018.schema.names

In [96]:
#Generamos los nombres de las columnas que nos interesan para cada asignatura y el peso del alumno

#Matematias
columnas_math = [col for col in columnas_disponibles_2018 if col.startswith('PV') and col.endswith('MATH')]

#Literatura
columnas_lectura = [col for col in columnas_disponibles_2018 if col.startswith('PV') and col.endswith('READ')]

#Ciencias
columnas_ciencias = [col for col in columnas_disponibles_2018 if col.startswith('PV') and col.endswith('SCIE')]


In [97]:
print(f"Columnas de matemáticas encontradas: {len(columnas_math)}")
print(f"Columnas de lectura encontradas: {len(columnas_lectura)}")
print(f"Columnas de ciencias encontradas: {len(columnas_ciencias)}")

Columnas de matemáticas encontradas: 10
Columnas de lectura encontradas: 10
Columnas de ciencias encontradas: 10


In [98]:
# Cargamos país, estrato, PV valores y peso del alumno.
columnas_interes = columnas_id + columnas_math + columnas_lectura + columnas_ciencias + ['W_FSTUWT']

#Hacemos la lectura de los datos de PV valores que nos interesan
df_pisa = pd.read_parquet(ruta_archivo_estudiantes_2018, columns=columnas_interes)

In [99]:
df_medias_math_pais_2018 = puntuaciones_pisa_pais(df_pisa, 'MATH', col_peso='W_FSTUWT')
df_medias_lectura_pais_2018 = puntuaciones_pisa_pais(df_pisa, 'READ', col_peso='W_FSTUWT')
df_medias_ciencias_pais_2018 = puntuaciones_pisa_pais(df_pisa, 'SCIE', col_peso='W_FSTUWT')

df_medias_math_estrato_2018 = puntuaciones_pisa_estratos(df_pisa, 'MATH', col_peso='W_FSTUWT')
df_medias_lectura_estrato_2018 = puntuaciones_pisa_estratos(df_pisa, 'READ', col_peso='W_FSTUWT')
df_medias_ciencias_estrato_2018 = puntuaciones_pisa_estratos(df_pisa, 'SCIE', col_peso='W_FSTUWT')

In [100]:
display(df_medias_math_pais_2018.head())
display(df_medias_lectura_pais_2018.head())
display(df_medias_ciencias_pais_2018.head())
display(df_medias_math_estrato_2018.head())
display(df_medias_lectura_estrato_2018.head())
display(df_medias_ciencias_estrato_2018.head())

,CNT,STRATUM,media_math_estrato
0,AUS,AUS0101,509.802734
1,AUS,AUS0102,507.289246
2,AUS,AUS0103,551.036804
3,AUS,AUS0204,493.826538
4,AUS,AUS0205,476.040222


,CNT,STRATUM,media_read_estrato
0,AUS,AUS0101,530.309509
1,AUS,AUS0102,528.143250
2,AUS,AUS0103,567.280945
3,AUS,AUS0204,506.625031
4,AUS,AUS0205,477.001953


,CNT,STRATUM,media_scie_estrato
0,AUS,AUS0101,527.919189
1,AUS,AUS0102,524.873474
2,AUS,AUS0103,567.823914
3,AUS,AUS0204,504.967346
4,AUS,AUS0205,481.300964


In [101]:
df_medias_math_pais_2018.to_parquet(ruta_medias_asignaturas/'pisa18_medias_math_pais.parquet', index=False)
df_medias_lectura_pais_2018.to_parquet(ruta_medias_asignaturas/'pisa18_medias_read_pais.parquet', index=False)
df_medias_ciencias_pais_2018.to_parquet(ruta_medias_asignaturas/'pisa18_medias_scie_pais.parquet', index=False)
df_medias_math_estrato_2018.to_parquet(ruta_medias_asignaturas/'pisa18_medias_math_estrato.parquet', index=False)
df_medias_lectura_estrato_2018.to_parquet(ruta_medias_asignaturas/'pisa18_medias_read_estrato.parquet', index=False)
df_medias_ciencias_estrato_2018.to_parquet(ruta_medias_asignaturas/'pisa18_medias_scie_estrato.parquet', index=False)

# Medias asignaturas PISA 2015 por país y estrato

In [102]:
#Cargamos todos los datos de estudiantes de 2015
ruta_archivo_estudiantes_2015 = ruta_datos_raw / 'pisa2015_estudiantes.parquet'
archivo_pisa_2015 = pq.ParquetFile(ruta_archivo_estudiantes_2015)
columnas_disponibles_2015 = archivo_pisa_2015.schema.names

In [103]:
#Generamos los nombres de las columnas que nos interesan para cada asignatura y el peso del alumno

#Matematias
columnas_math = [col for col in columnas_disponibles_2015 if col.startswith('PV') and col.endswith('MATH')]

#Literatura
columnas_lectura = [col for col in columnas_disponibles_2015 if col.startswith('PV') and col.endswith('READ')]

#Ciencias
columnas_ciencias = [col for col in columnas_disponibles_2015 if col.startswith('PV') and col.endswith('SCIE')]


In [104]:
print(f"Columnas de matemáticas encontradas: {len(columnas_math)}")
print(f"Columnas de lectura encontradas: {len(columnas_lectura)}")
print(f"Columnas de ciencias encontradas: {len(columnas_ciencias)}")

Columnas de matemáticas encontradas: 10
Columnas de lectura encontradas: 10
Columnas de ciencias encontradas: 10


In [105]:
# Cargamos país, estrato, PV valores y peso del alumno.
columnas_interes = columnas_id + columnas_math + columnas_lectura + columnas_ciencias + ['W_FSTUWT']

#Hacemos la lectura de los datos de PV valores que nos interesan
df_pisa = pd.read_parquet(ruta_archivo_estudiantes_2015, columns=columnas_interes)

In [106]:
df_medias_math_pais_2015 = puntuaciones_pisa_pais(df_pisa, 'MATH', col_peso='W_FSTUWT')
df_medias_lectura_pais_2015 = puntuaciones_pisa_pais(df_pisa, 'READ', col_peso='W_FSTUWT')
df_medias_ciencias_pais_2015 = puntuaciones_pisa_pais(df_pisa, 'SCIE', col_peso='W_FSTUWT')

df_medias_math_estrato_2015 = puntuaciones_pisa_estratos(df_pisa, 'MATH', col_peso='W_FSTUWT')
df_medias_lectura_estrato_2015 = puntuaciones_pisa_estratos(df_pisa, 'READ', col_peso='W_FSTUWT')
df_medias_ciencias_estrato_2015 = puntuaciones_pisa_estratos(df_pisa, 'SCIE', col_peso='W_FSTUWT')

In [107]:
display(df_medias_math_pais_2015.head())
display(df_medias_lectura_pais_2015.head())
display(df_medias_ciencias_pais_2015.head())
display(df_medias_math_estrato_2015.head())
display(df_medias_lectura_estrato_2015.head())
display(df_medias_ciencias_estrato_2015.head())

,CNT,STRATUM,media_math_estrato
0,AUS,AUS0101,509.460999
1,AUS,AUS0102,471.223145
2,AUS,AUS0103,488.917419
3,AUS,AUS0104,465.288391
4,AUS,AUS0105,551.424927


,CNT,STRATUM,media_read_estrato
0,AUS,AUS0101,517.610779
1,AUS,AUS0102,499.196136
2,AUS,AUS0103,498.672791
3,AUS,AUS0104,474.057007
4,AUS,AUS0105,565.265137


,CNT,STRATUM,media_scie_estrato
0,AUS,AUS0101,532.446960
1,AUS,AUS0102,509.502960
2,AUS,AUS0103,508.337860
3,AUS,AUS0104,487.153046
4,AUS,AUS0105,573.402283


In [108]:
df_medias_math_pais_2015.to_parquet(ruta_medias_asignaturas/'pisa15_medias_math_pais.parquet', index=False)
df_medias_lectura_pais_2015.to_parquet(ruta_medias_asignaturas/'pisa15_medias_read_pais.parquet', index=False)
df_medias_ciencias_pais_2015.to_parquet(ruta_medias_asignaturas/'pisa15_medias_scie_pais.parquet', index=False)
df_medias_math_estrato_2015.to_parquet(ruta_medias_asignaturas/'pisa15_medias_math_estrato.parquet', index=False)
df_medias_lectura_estrato_2015.to_parquet(ruta_medias_asignaturas/'pisa15_medias_read_estrato.parquet', index=False)
df_medias_ciencias_estrato_2015.to_parquet(ruta_medias_asignaturas/'pisa15_medias_scie_estrato.parquet', index=False)

# Medias asignaturas PISA 2012 por país y estrato

In [109]:
#Cargamos todos los datos de estudiantes de 2012
ruta_archivo_estudiantes_2012 = ruta_datos_raw / 'pisa2012_estudiantes.parquet'
archivo_pisa_2012 = pq.ParquetFile(ruta_archivo_estudiantes_2012)
columnas_disponibles_2012 = archivo_pisa_2012.schema.names

In [110]:
#Generamos los nombres de las columnas que nos interesan para cada asignatura y el peso del alumno

#Matematias
columnas_math = [col for col in columnas_disponibles_2012 if col.startswith('PV') and col.endswith('MATH')]

#Literatura
columnas_lectura = [col for col in columnas_disponibles_2012 if col.startswith('PV') and col.endswith('READ')]

#Ciencias
columnas_ciencias = [col for col in columnas_disponibles_2012 if col.startswith('PV') and col.endswith('SCIE')]


In [111]:
print(f"Columnas de matemáticas encontradas: {len(columnas_math)}")
print(f"Columnas de lectura encontradas: {len(columnas_lectura)}")
print(f"Columnas de ciencias encontradas: {len(columnas_ciencias)}")

Columnas de matemáticas encontradas: 5
Columnas de lectura encontradas: 5
Columnas de ciencias encontradas: 5


In [112]:
# Cargamos país, estrato, PV valores y peso del alumno.
columnas_interes = columnas_id + columnas_math + columnas_lectura + columnas_ciencias + ['W_FSTUWT']

#Hacemos la lectura de los datos de PV valores que nos interesan
df_pisa = pd.read_parquet(ruta_archivo_estudiantes_2012, columns=columnas_interes)

In [113]:
df_medias_math_pais_2012 = puntuaciones_pisa_pais(df_pisa, 'MATH', col_peso='W_FSTUWT')
df_medias_lectura_pais_2012 = puntuaciones_pisa_pais(df_pisa, 'READ', col_peso='W_FSTUWT')
df_medias_ciencias_pais_2012 = puntuaciones_pisa_pais(df_pisa, 'SCIE', col_peso='W_FSTUWT')

df_medias_math_estrato_2012 = puntuaciones_pisa_estratos(df_pisa, 'MATH', col_peso='W_FSTUWT')
df_medias_lectura_estrato_2012 = puntuaciones_pisa_estratos(df_pisa, 'READ', col_peso='W_FSTUWT')
df_medias_ciencias_estrato_2012 = puntuaciones_pisa_estratos(df_pisa, 'SCIE', col_peso='W_FSTUWT')

In [114]:
display(df_medias_math_pais_2012.head())
display(df_medias_lectura_pais_2012.head())
display(df_medias_ciencias_pais_2012.head())
display(df_medias_math_estrato_2012.head())
display(df_medias_lectura_estrato_2012.head())
display(df_medias_ciencias_estrato_2012.head())

,CNT,STRATUM,media_math_estrato
0,AUS,AUS0101,510.778564
1,AUS,AUS0102,499.952454
2,AUS,AUS0103,580.811646
3,AUS,AUS0204,517.748901
4,AUS,AUS0205,496.620422


,CNT,STRATUM,media_read_estrato
0,AUS,AUS0101,542.527710
1,AUS,AUS0102,495.108185
2,AUS,AUS0103,580.720459
3,AUS,AUS0204,523.719360
4,AUS,AUS0205,498.356598


,CNT,STRATUM,media_scie_estrato
0,AUS,AUS0101,537.181213
1,AUS,AUS0102,511.018036
2,AUS,AUS0103,593.751953
3,AUS,AUS0204,533.861877
4,AUS,AUS0205,512.010315


In [115]:
df_medias_math_pais_2012.to_parquet(ruta_medias_asignaturas/'pisa12_medias_math_pais.parquet', index=False)
df_medias_lectura_pais_2012.to_parquet(ruta_medias_asignaturas/'pisa12_medias_read_pais.parquet', index=False)
df_medias_ciencias_pais_2012.to_parquet(ruta_medias_asignaturas/'pisa12_medias_scie_pais.parquet', index=False)
df_medias_math_estrato_2012.to_parquet(ruta_medias_asignaturas/'pisa12_medias_math_estrato.parquet', index=False)
df_medias_lectura_estrato_2012.to_parquet(ruta_medias_asignaturas/'pisa12_medias_read_estrato.parquet', index=False)
df_medias_ciencias_estrato_2012.to_parquet(ruta_medias_asignaturas/'pisa12_medias_scie_estrato.parquet', index=False)

# Limpieza de memoria

In [ ]:


# Liberamos los datos de estudiantes y variables temporales, conservando las medias calculadas.
variables_temporales = [
    'df_pisa', 'archivo_pisa_2022', 'archivo_pisa_2018', 'archivo_pisa_2015', 'archivo_pisa_2012',
    'columnas_disponibles_2022', 'columnas_disponibles_2018', 'columnas_disponibles_2015', 'columnas_disponibles_2012',
    'columnas_math', 'columnas_lectura', 'columnas_ciencias', 'columnas_id', 'columnas_interes',
    'columnas_math_2018', 'columnas_lectura_2018', 'columnas_ciencias_2018', 'columnas_id_2018', 'columnas_interes_2018',
    'columnas_math_2015', 'columnas_lectura_2015', 'columnas_ciencias_2015', 'columnas_id_2015', 'columnas_interes_2015',
    'columnas_math_2012', 'columnas_lectura_2012', 'columnas_ciencias_2012', 'columnas_id_2012', 'columnas_interes_2012',
]
for variable in variables_temporales:
    if variable in globals():
        del globals()[variable]

del variables_temporales
objetos_liberados = gc.collect()
print(f"Elementos liberados: {objetos_liberados}")

Objetos recogidos por el recolector: 26


# Creación de df global

In [121]:
asignaturas_medias = {
    'math': 'media_math_estrato',
    'read': 'media_read_estrato',
    'scie': 'media_scie_estrato',
}
years = [2012, 2015, 2018, 2022]
df_per_year = []

for anio in years:
    df_anual = None
    for asignatura, columna_media in asignaturas_medias.items():
        ruta_media = ruta_medias_asignaturas / f'pisa{str(anio)[-2:]}_medias_{asignatura}_estrato.parquet'
        df_asignatura = pd.read_parquet(
            ruta_media,
            columns=['CNT', 'STRATUM', columna_media],
        )
        if df_anual is None:
            df_anual = df_asignatura
        else:
            df_anual = df_anual.merge(
                df_asignatura,
                on=['CNT', 'STRATUM'],
                how='outer',
                validate='one_to_one',
            )

    df_anual['year'] = anio
    df_per_year.append(df_anual)

df_total = (
    pd.concat(df_per_year, ignore_index=True)
    [['CNT', 'STRATUM', 'year', 'media_math_estrato', 'media_read_estrato', 'media_scie_estrato']]
    .sort_values(['year', 'CNT', 'STRATUM'])
    .reset_index(drop=True)
)

display(df_total.head())

,CNT,STRATUM,year,media_math_estrato,media_read_estrato,media_scie_estrato
0,AUS,AUS0101,2012,510.778564,542.527710,537.181213
1,AUS,AUS0102,2012,499.952454,495.108185,511.018036
2,AUS,AUS0103,2012,580.811646,580.720459,593.751953
3,AUS,AUS0204,2012,517.748901,523.719360,533.861877
4,AUS,AUS0205,2012,496.620422,498.356598,512.010315


In [ ]:
ruta_df_total = ruta_medias_asignaturas / 'pisa_medias_asignaturas_estrato_anual.parquet'
df_total.to_parquet(ruta_df_total, index=False)

asignaturas_medias_pais = {
    'math': 'media_math_pais',
    'read': 'media_read_pais',
    'scie': 'media_scie_pais',
}
df_per_year_pais = []

for anio in years:
    df_anual_pais = None
    for asignatura, columna_media in asignaturas_medias_pais.items():
        ruta_media = ruta_medias_asignaturas / f'pisa{str(anio)[-2:]}_medias_{asignatura}_pais.parquet'
        df_asignatura = pd.read_parquet(ruta_media, columns=['CNT', columna_media])
        if df_anual_pais is None:
            df_anual_pais = df_asignatura
        else:
            df_anual_pais = df_anual_pais.merge(
                df_asignatura,
                on='CNT',
                how='outer',
                validate='one_to_one',
            )

    df_anual_pais['year'] = anio
    df_per_year_pais.append(df_anual_pais)

df_total_pais = (
    pd.concat(df_per_year_pais, ignore_index=True)
    [['CNT', 'year', 'media_math_pais', 'media_read_pais', 'media_scie_pais']]
    .sort_values(['year', 'CNT'])
    .reset_index(drop=True)
)

display(df_total_pais.head())
ruta_df_total_pais = ruta_medias_asignaturas / 'pisa_medias_asignaturas_pais_anual.parquet'
df_total_pais.to_parquet(ruta_df_total_pais, index=False)